<a href="https://colab.research.google.com/github/smargetic/Natural_Language_Processing/blob/main/Machine_Translation/Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiments: comparing configurations and keeping the best model

The model itself is defined in `Translation_Model_Pytorch.ipynb`. This notebook uses it to answer *which settings translate best?*

**How it works**
1. Load every function and the default `CONFIG` from the model notebook.
2. List the experiments to try. Each one is a name plus the few settings it changes.
3. Train each experiment and score it on the **validation** sentences (BLEU). Results are saved to `experiments/results.csv` after every experiment, so nothing is lost if the run stops. Finished experiments are skipped when you run again.
4. Copy the experiment with the best validation BLEU to `models/best_model.pt`.
5. Score that one model on the **test** sentences.

**Why validation for choosing and test only at the end.** Picking the winner by its test score would favour whichever model happened to fit the test sentences, and the reported number would be too optimistic. The test sentences stay unseen until the very end, so the final score is an honest estimate for new sentences.

**Time.** On a laptop CPU an LSTM experiment takes about 50 minutes with the default settings, and a Transformer experiment is faster. In Colab, choose **Runtime → Change runtime type → GPU**.

**Sections**
1. Load the model notebook
2. Settings for all experiments
3. The experiments to run
4. A look at the data and the segmentation methods
5. Run the experiments
6. Compare the results
7. Keep the best model
8. Final test of the best model
9. Try your own sentences

## 1. Load the model notebook

`%run` runs `Translation_Model_Pytorch.ipynb` from top to bottom. It only defines functions and settings, so it takes a few seconds.

In [ ]:
import os
import sys
import json
import copy
import shutil

#in Colab: mount Google Drive and move into the project folder first, so %run can find the model notebook
if 'google.colab' in sys.modules:
  from google.colab import drive
  drive.mount('/content/drive')
  os.chdir('/content/drive/My Drive/Coding Projects/Machine Translation')   #<- change this if the folder moves

%run Translation_Model_Pytorch.ipynb

## 2. Settings for all experiments

Every experiment starts from `BASE_CONFIG`: the defaults from the model notebook, plus anything changed here. Change a setting here to change it for **every** experiment, e.g. more data for all of them.

In [ ]:
EXPERIMENTS_FOLDER = 'experiments'                                 #one subfolder per experiment, plus results.csv
RESULTS_FILE = os.path.join(EXPERIMENTS_FOLDER, 'results.csv')
BEST_MODEL_FOLDER = 'models'
BEST_MODEL_PATH = os.path.join(BEST_MODEL_FOLDER, 'best_model.pt')
BEST_MODEL_INFO = os.path.join(BEST_MODEL_FOLDER, 'best_model_info.json')

BASE_CONFIG = copy.deepcopy(CONFIG)
# BASE_CONFIG['sample_frac'] = 0.2       #example: 20% of the data for every experiment (best with a GPU)
# BASE_CONFIG['n_epochs'] = 15

## 3. The experiments to run

Each line is a **name** and the **settings it changes** from `BASE_CONFIG`; every other setting stays the same. That way two experiments differ only in what you want to compare.

- To add an experiment, add a line. Any key from `CONFIG` can be changed.
- Experiments already in `experiments/results.csv` are skipped. To run one again, delete its row from that file, or give it a new name.

The list below compares the four Italian segmentation methods with everything else equal, then tries a bidirectional LSTM and a Transformer with BPE.

In [ ]:
EXPERIMENTS = [
  #name                  settings changed from BASE_CONFIG
  ('lstm_words',         {'segmentation': {'english': 'words', 'italian': 'words'}}),
  ('lstm_bpe',           {'segmentation': {'english': 'words', 'italian': 'bpe'}}),
  ('lstm_stem_bpe',      {'segmentation': {'english': 'words', 'italian': 'stem+bpe'}}),
  ('lstm_lemma_tags',    {'segmentation': {'english': 'words', 'italian': 'lemma+tags'}}),
  ('bilstm_bpe',         {'segmentation': {'english': 'words', 'italian': 'bpe'}, 'bidirectional': True}),
  ('transformer_bpe',    {'segmentation': {'english': 'words', 'italian': 'bpe'}, 'node_type': 'transformer',
                          'enc_n_layers': 3, 'dec_n_layers': 3, 'learning_rate': 0.0005}),
]


#BASE_CONFIG + one experiment's changes -> that experiment's full settings (and its own checkpoint file)
def make_config(name, changes):
  unknown = set(changes) - set(BASE_CONFIG)
  if unknown:                                  #catches typos such as 'n_epoch'
    raise KeyError('experiment {!r} changes unknown settings: {}'.format(name, sorted(unknown)))
  config = copy.deepcopy(BASE_CONFIG)
  config.update(copy.deepcopy(changes))
  config['checkpoint_path'] = os.path.join(EXPERIMENTS_FOLDER, name, 'model.pt')
  return config


names = [name for name, _ in EXPERIMENTS]
assert len(names) == len(set(names)), 'experiment names must be unique'
for name, changes in EXPERIMENTS:
  make_config(name, changes)                   #check every experiment's settings before training anything
print('{} experiments defined: {}'.format(len(EXPERIMENTS), ', '.join(names)))

## 4. A look at the data and the segmentation methods

Optional, and quick compared with training. It shows one sentence pair end to end, how much a batch holds, and how the four segmentation methods split the Italian training data:
- **vocab**: number of distinct tokens the model must learn
- **tokens/word**: how much longer sentences get (longer means slower training and more decoding steps)
- **test `<unk>`**: share of test tokens the vocabulary can't represent

`'lemma+tags'` is skipped if spaCy or the Italian model isn't installed.

In [ ]:
train_df, val_df, test_df, eng_vocab, it_vocab, train_loader, val_loader, test_loader = prepare_data(BASE_CONFIG)

#one example end to end
example = train_df.iloc[0]
print('\nExample pair:')
for name, sentence, words, vocab in [('English', example['eng_sentence'], example['eng_words'], eng_vocab),
                                     ('Italian', example['it_sentence'], example['it_words'], it_vocab)]:
  print('  {}: {}'.format(name, sentence))
  print('    tokens: ', vocab.to_tokens(words))
  print('    indices:', vocab.encode(vocab.to_tokens(words)))

#one batch
eng_batch, it_batch = next(iter(train_loader))
print('\nOne batch: English', tuple(eng_batch.shape), ' Italian', tuple(it_batch.shape), '  (B, S) and (B, T)')

In [ ]:
compare_words = ['andiamo', 'andremo', 'andrebbero', 'parliamo', 'dimmelo', 'sulla', 'case', 'il']
rows = []
for method in SEGMENTERS:
  try:
    segmenter = SEGMENTERS[method]('italian', n_merges=BASE_CONFIG['bpe_merges']).fit(train_df['it_words'])
  except (ImportError, OSError) as error:        #library or language model not installed
    print('{} skipped: {}'.format(method, error))
    continue
  vocab = build_vocabulary(train_df['it_words'], segmenter, BASE_CONFIG['min_word_count'])
  test_tokens = [token for words in test_df['it_words'] for token in segmenter.to_tokens(words)]
  n_test_words = sum(len(words) for words in test_df['it_words'])
  row = {'method': method,
         'vocab': len(vocab),
         'tokens/word': round(len(test_tokens) / n_test_words, 2),
         'test <unk> %': round(100 * sum(token not in vocab.word_to_index for token in test_tokens) / len(test_tokens), 2)}
  for word in compare_words:
    row[word] = ' '.join(segmenter.to_tokens([word]))
  rows.append(row)

display(pd.DataFrame(rows).set_index('method').T)

## 5. Run the experiments

Each experiment calls `run_experiment(config)` from the model notebook: prepare the data, build a fresh model, train it, keep the epoch with the lowest validation loss, and compute validation BLEU. The random seed is reset before each experiment, so results don't depend on the order of the list.

This is the long step. You can stop it at any time; finished experiments are already saved.

In [ ]:
#results saved by earlier runs (an empty table the first time)
def load_results():
  if os.path.exists(RESULTS_FILE):
    return pd.read_csv(RESULTS_FILE)
  return pd.DataFrame(columns=['name', 'changes'])


results = load_results()

for name, changes in EXPERIMENTS:
  if name in set(results['name']):
    print('Skipping {} (already in {})'.format(name, RESULTS_FILE))
    continue

  print('\n' + '=' * 80)
  print('Experiment: {}    changes: {}'.format(name, changes))
  print('=' * 80)

  random.seed(SEED)
  torch.manual_seed(SEED)
  row = {'name': name, 'changes': json.dumps(changes)}
  row.update(run_experiment(make_config(name, changes)))

  #save after every experiment
  results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
  os.makedirs(EXPERIMENTS_FOLDER, exist_ok=True)
  results.to_csv(RESULTS_FILE, index=False)
  print('\n{}: validation BLEU {:.1f}  (saved to {})'.format(name, row['val_bleu'], RESULTS_FILE))

## 6. Compare the results

Ranked by **validation BLEU**, which is computed on whole words and can be compared across all experiments. Validation **loss** can only be compared between experiments that use the same segmentation, because each segmentation predicts different tokens.

In [ ]:
results = load_results()
ranked = results.sort_values('val_bleu', ascending=False).reset_index(drop=True)
display(ranked[['name', 'val_bleu', 'val_loss', 'best_epoch', 'italian_vocab', 'minutes', 'changes']])

#bar chart of validation BLEU (skipped if matplotlib isn't installed)
try:
  import matplotlib.pyplot as plt
  ax = ranked.plot.barh(x='name', y='val_bleu', legend=False, figsize=(7, 0.5 * len(ranked) + 1), color='#4C78A8')
  ax.invert_yaxis()
  ax.set_xlabel('Validation BLEU (higher is better)')
  ax.set_ylabel('')
  for i, value in enumerate(ranked['val_bleu']):
    ax.text(value, i, ' {:.1f}'.format(value), va='center')
  plt.tight_layout()
  plt.show()
except ImportError:
  pass

## 7. Keep the best model

The experiment with the highest validation BLEU is copied to `models/best_model.pt`, with a small `best_model_info.json` describing it. The checkpoint holds everything needed to translate later (weights, settings, vocabularies, segmenter state), so `load_checkpoint('models/best_model.pt')` is enough to use it.

In [ ]:
best = ranked.iloc[0]
os.makedirs(BEST_MODEL_FOLDER, exist_ok=True)
shutil.copy(best['checkpoint'], BEST_MODEL_PATH)

best_info = {'name': best['name'],
             'changes': json.loads(best['changes']),
             'val_bleu': float(best['val_bleu']),
             'best_epoch': int(best['best_epoch'])}
with open(BEST_MODEL_INFO, 'w') as f:
  json.dump(best_info, f, indent=2)

print('Best experiment: {} (validation BLEU {:.1f})'.format(best['name'], best['val_bleu']))
print('Saved to', BEST_MODEL_PATH)

## 8. Final test of the best model

The test sentences are used here for the first and only time. The data is prepared with the best experiment's settings, so the split is the same as during training.

In [ ]:
best_config = make_config(best['name'], json.loads(best['changes']))
model, eng_vocab, it_vocab = load_checkpoint(BEST_MODEL_PATH)
_, _, test_df, _, _, _, _, _ = prepare_data(best_config)
test_loader = DataLoader(TranslationDataset(test_df, eng_vocab, it_vocab), batch_size=best_config['batch_size'], shuffle=False, collate_fn=pad_batch)

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
test_loss = run_epoch(model, test_loader, loss_fn)
print('Test loss {:.3f} | test perplexity {:.1f}'.format(test_loss, math.exp(test_loss)))

for width in [1, best_config['beam_width']]:
  bleu, examples = evaluate_bleu(model, test_df, eng_vocab, it_vocab, n_sentences=best_config['bleu_sentences'], beam_width=width,
                                 max_length=best_config['max_output_words'], norm_method=best_config['norm_method'])
  print('Test BLEU with beam width {}: {:.1f}'.format(width, bleu))

display(examples.head(15))

## 9. Try your own sentences

In [ ]:
my_sentences = [
  'I love you.',
  'What time is it?',
  'The cat is sleeping on the bed.',
  'We are going to the beach tomorrow.',
]

for sentence in my_sentences:
  print('{}  ->  {}'.format(sentence, translate(model, sentence, eng_vocab, it_vocab, best_config['beam_width'],
                                                best_config['max_output_words'], best_config['norm_method'])))